In [ ]:
import os


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [ ]:
dir_path = "/scratch/yag1/ego4d_data/v2/clips"

mp3_files = [f for f in os.listdir(dir_path) if f.endswith(".mp3")]

print(f"Number of mp3 files: {len(mp3_files)}")

In [ ]:
from transformers import ClapModel, ClapProcessor


In [ ]:
model = ClapModel.from_pretrained("laion/clap-htsat-fused").to(0)
processor = ClapProcessor.from_pretrained("laion/clap-htsat-fused")


In [ ]:
# !pip install pydub
# !pip install librosa

In [ ]:
from pydub import AudioSegment
import json
import torch
from tqdm import tqdm
import numpy as np
import librosa

In [ ]:
target_sr = 48000 # This is the sample rate that the model expects. We will resample the audio to this rate if it's not already at this rate.

In [ ]:
# Load all segment JSONs once
segments_by_duration = {}
for i in range(1, 11):
    with open(f"{i}_seconds_segments.json") as f:
        segments_by_duration[i] = json.load(f)

for mp3_file in tqdm(mp3_files):
    mp3_name = os.path.splitext(mp3_file)[0]
    os.makedirs(f"{dir_path}/{mp3_name}/audio_embeddings", exist_ok=True)

    mp3_path = os.path.join(dir_path, mp3_file)
    audio = AudioSegment.from_mp3(mp3_path)

    for i in range(1, 11):
        segments = segments_by_duration[i][f"{dir_path}/{mp3_name}.mp4"]  # fix extension

        for idx, segment in enumerate(segments):
            start_time = segment[0] * 1000
            end_time = segment[1] * 1000
            audio_sample = audio[start_time:end_time]
            audio_array = np.array(audio_sample.get_array_of_samples()).astype(np.float32)
            audio_array /= np.iinfo(audio_sample.array_type).max

            if audio_sample.frame_rate != target_sr:
                audio_array = librosa.resample(audio_array, orig_sr=audio_sample.frame_rate, target_sr=target_sr)

            seg_id = f"{i}_seconds_segment_{idx}"
            inputs = processor(audio=audio_array, sampling_rate=target_sr, return_tensors="pt").to(0)

            with torch.no_grad():
                model_output = model.get_audio_features(**inputs)
            # print(model_output.pooler_output[0].shape)
            torch.save(model_output.pooler_output[0].cpu(), f"{dir_path}/{mp3_name}/audio_embeddings/{seg_id}.pt")

    print(f"Finished saving embeddings for {mp3_file}")

In [ ]:
# !pip uninstall transformers -y

In [ ]:
# !pip install transformers